# Demo - Performance Comparison (CIFAR10)

## 1. Load CIAFR10

In [ ]:
# robustbench upload zip need unzip only need run once

In [1]:
import zipfile
import os

# Define the path of the ZIP file
zip_file_path = '/content/robustbench.zip'

# Get the directory where the ZIP file is located
extract_path = os.path.dirname(zip_file_path)

try:
    # Open the ZIP file
    with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
        # Extract the ZIP file to the original directory
        zip_ref.extractall(extract_path)
    print("Unzipped successfully!")
except FileNotFoundError:
    print(f"Error: File {zip_file_path} not found.")
except zipfile.BadZipFile:
    print(f"Error: {zip_file_path} is not a valid ZIP file.")
except Exception as e:
    print(f"An unknown error occurred: {e}")


Unzipped successfully!


In [2]:
from robustbench.data import load_cifar10
from robustbench.utils import clean_accuracy
from utils import l2_distance
from robustbench.model_zoo.enums import BenchmarkDataset, ThreatModel
from typing import Dict, Optional, Union
from pathlib import Path
import torch
from torch import nn
images, labels = load_cifar10(n_examples=10)
device = "cuda"

## 2. Standard Accuracy

In [3]:
import torch

# 只在第一次运行时 patch
if not hasattr(torch, '_original_load'):
    torch._original_load = torch.load  # 保存原始函数

def patched_torch_load(*args, **kwargs):
    if 'weights_only' not in kwargs:
        kwargs['weights_only'] = False
    return torch._original_load(*args, **kwargs)  # 调用原始函数

torch.load = patched_torch_load

In [4]:
import argparse
import dataclasses
import json
import math
import os
import warnings
from collections import OrderedDict
from pathlib import Path
from typing import Dict, Optional, Union

import torch
from torch import nn

from robustbench.model_zoo import model_dicts as all_models
from robustbench.model_zoo.enums import BenchmarkDataset, ThreatModel


ACC_FIELDS = {
    ThreatModel.corruptions: "corruptions_acc",
    ThreatModel.L2: ("external", "autoattack_acc"),
    ThreatModel.Linf: ("external", "autoattack_acc")
}


def load_model(model_name: str,
               model_dir: Union[str, Path] = './models',
               dataset: Union[str, BenchmarkDataset] = BenchmarkDataset.cifar_10,
               threat_model: Union[str, ThreatModel] = ThreatModel.Linf,
               norm: Optional[str] = None) -> nn.Module:
    """Loads a model from the model_zoo.

     The model is trained on the given ``dataset``, for the given ``threat_model``.

    :param model_name: The name used in the model zoo.
    :param model_dir: The base directory where the models are saved.
    :param dataset: The dataset on which the model is trained.
    :param threat_model: The threat model for which the model is trained.
    :param norm: Deprecated argument that can be used in place of ``threat_model``. If specified, it
      overrides ``threat_model``

    :return: A ready-to-used trained model.
    """
    dataset_: BenchmarkDataset = BenchmarkDataset(dataset)
    if norm is None:
        threat_model_: ThreatModel = ThreatModel(threat_model)
    else:
        threat_model_ = ThreatModel(norm)
        warnings.warn(
            "`norm` has been deprecated and will be removed in a future version.",
            DeprecationWarning)

    model_dir_ = Path(model_dir) / dataset_.value / threat_model_.value
    model_path = model_dir_ / f'{model_name}.pt'

    models = all_models[dataset_][threat_model_]

    model = models[model_name]['model']()

    def rm_substr_from_state_dict(state_dict, substr):
        new_state_dict = OrderedDict()
        for key in state_dict.keys():
            if substr in key:  # to delete prefix 'module.' if it exists
                new_key = key[len(substr):]
                new_state_dict[new_key] = state_dict[key]
            else:
                new_state_dict[key] = state_dict[key]
        return new_state_dict

    def add_substr_to_state_dict(state_dict, substr):
        new_state_dict = OrderedDict()
        for k, v in state_dict.items():
            new_state_dict[substr + k] = v
        return new_state_dict

    def _safe_load_state_dict(model: nn.Module, model_name: str,
                              state_dict: Dict[str, torch.Tensor],
                              dataset_: BenchmarkDataset) -> nn.Module:
        known_failing_models = {
            "Andriushchenko2020Understanding", "Augustin2020Adversarial",
            "Engstrom2019Robustness", "Pang2020Boosting", "Rice2020Overfitting",
            "Rony2019Decoupling", "Wong2020Fast", "Hendrycks2020AugMix_WRN",
            "Hendrycks2020AugMix_ResNeXt", "Kireev2021Effectiveness_Gauss50percent",
            "Kireev2021Effectiveness_AugMixNoJSD", "Kireev2021Effectiveness_RLAT",
            "Kireev2021Effectiveness_RLATAugMixNoJSD", "Kireev2021Effectiveness_RLATAugMixNoJSD",
            "Kireev2021Effectiveness_RLATAugMix", "Chen2020Efficient",
            "Wu2020Adversarial", "Augustin2020Adversarial_34_10",
            "Augustin2020Adversarial_34_10_extra", "Diffenderfer2021Winning_LRR",
            "Diffenderfer2021Winning_LRR_CARD_Deck", "Diffenderfer2021Winning_Binary",
            "Diffenderfer2021Winning_Binary_CARD_Deck"
        }

        failure_messages = ['Missing key(s) in state_dict: "mu", "sigma".',
                            'Unexpected key(s) in state_dict: "model_preact_hl1.1.weight"',
                            'Missing key(s) in state_dict: "normalize.mean", "normalize.std"',
                            'Unexpected key(s) in state_dict: "conv1.scores"']

        try:
            model.load_state_dict(state_dict, strict=True)
        except RuntimeError as e:
            if (model_name in known_failing_models or dataset_ == BenchmarkDataset.imagenet
                    ) and any([msg in str(e) for msg in failure_messages]):
                model.load_state_dict(state_dict, strict=False)
            else:
                raise e

        return model

    # 直接加载本地模型文件，移除下载逻辑
    checkpoint = torch.load(model_path, map_location=torch.device('cpu'))

    if 'Kireev2021Effectiveness' in model_name or model_name == 'Andriushchenko2020Understanding':
        checkpoint = checkpoint['last']  # we take the last model (choices: 'last', 'best')
    try:
        state_dict = rm_substr_from_state_dict(checkpoint['state_dict'], 'module.')
        state_dict = rm_substr_from_state_dict(state_dict, 'model.')
    except:
        state_dict = rm_substr_from_state_dict(checkpoint, 'module.')
        state_dict = rm_substr_from_state_dict(state_dict, 'model.')

    if dataset_ == BenchmarkDataset.imagenet:
        state_dict = add_substr_to_state_dict(state_dict, 'model.')

    model = _safe_load_state_dict(model, model_name, state_dict, dataset_)

    return model.eval()


In [5]:
def clean_accuracy(model, images, labels, device):
    with torch.no_grad():
        images = images.to(device)
        labels = labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        correct = (predicted == labels).sum().item()
        total = labels.size(0)
        return correct / total

In [6]:
import torch

model_path = '/content/models/cifar10/Linf/Standard.pt'  # 验证是否上传成功
try:
    checkpoint = torch.load(model_path, map_location=torch.device('cuda'))
    print("文件加载成功。")
except Exception as e:
    print(f"加载文件时出错: {e}")

文件加载成功。


In [7]:
# model_list = ['Standard', 'Wong2020Fast', 'Rice2020Overfitting']
model_list = ['Standard', 'Wong2020Fast']
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

images = images.to(device)
labels = labels.to(device)

for model_name in model_list:
    model = load_model(model_name, norm='Linf').to(device)
    acc = clean_accuracy(model, images.to(device), labels.to(device),device)
    print('Model: {}'.format(model_name))
    print('- Standard Acc: {}'.format(acc))

Model: Standard
- Standard Acc: 1.0
Model: Wong2020Fast
- Standard Acc: 1.0


## 3. Comparison with Foolbox and ART

In [8]:
!pip install foolbox

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 68.4 MB/s eta 0:00:00


In [9]:
!pip install art

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.6/608.6 kB 46.0 MB/s eta 0:00:00


In [11]:
!pip install --upgrade adversarial-robustness-toolbox

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 59.9 MB/s eta 0:00:00


In [3]:
!pip install torchattacks

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 59.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 47.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [1]:
import datetime
import numpy as np
import warnings

warnings.filterwarnings(action='ignore')

import torch
import torch.nn as nn
import torch.optim as optim

# https://github.com/bethgelab/foolbox
import foolbox as fb
print("foolbox %s"%(fb.__version__))

# https://github.com/IBM/adversarial-robustness-toolbox
import art
import art.attacks.evasion as evasion
#from art.classifiers import PyTorchClassifier
from art.estimators.classification.pytorch import PyTorchClassifier
print("art %s"%(art.__version__))

import sys
sys.path.insert(0, '..')
# https://github.com/Harry24k/adversarial-attacks-pytorch
import torchattacks
print("torchattacks %s"%(torchattacks.__version__))

foolbox 3.3.4
art 1.19.1
torchattacks 3.5.1


## 3.1. Linf

### FGSM

In [12]:
for model_name in model_list:
    print('Model: {}'.format(model_name))
    model = load_model(model_name, norm='Linf').to(device)

    print("- Torchattacks")
    atk = torchattacks.FGSM(model, eps=8/255)
    start = datetime.datetime.now()
    adv_images = atk(images, labels)
    end = datetime.datetime.now()
    acc = clean_accuracy(model, adv_images, labels,device)
    print('- Robust Acc: {} ({} ms)'.format(acc, int((end-start).total_seconds()*1000)))

    print("- Foolbox")
    fmodel = fb.PyTorchModel(model, bounds=(0, 1))
    atk = fb.attacks.LinfFastGradientAttack(random_start=False)
    start = datetime.datetime.now()
    _, adv_images, _ = atk(fmodel, images.to('cuda:0'), labels.to('cuda:0'), epsilons=8/255)
    end = datetime.datetime.now()
    acc = clean_accuracy(model, adv_images, labels,device)
    print('- Robust Acc: {} ({} ms)'.format(acc, int((end-start).total_seconds()*1000)))

    print("- ART")
    classifier = PyTorchClassifier(model=model, clip_values=(0, 1),
                                   loss=nn.CrossEntropyLoss(),
                                   optimizer=optim.Adam(model.parameters(), lr=0.01),
                                   input_shape=(3, 32, 32), nb_classes=10)
    atk = evasion.FastGradientMethod(norm=np.inf, batch_size=50,
                                     estimator=classifier, eps=8/255)
    start = datetime.datetime.now()
    #adv_images = torch.tensor(atk.generate(images, labels)).to(device)
    adv_images_np = atk.generate(images.cpu().numpy(), labels.cpu().numpy())
    adv_images = torch.from_numpy(adv_images_np).to(device)
    end = datetime.datetime.now()
    acc = clean_accuracy(model, adv_images, labels,device)
    print('- Robust Acc: {} ({} ms)'.format(acc, int((end-start).total_seconds()*1000)))

    print()

Model: Standard
- Torchattacks
- Robust Acc: 0.4 (12 ms)
- Foolbox
- Robust Acc: 0.4 (15 ms)
- ART
- Robust Acc: 0.4 (75 ms)

Model: Wong2020Fast
- Torchattacks
- Robust Acc: 0.6 (10 ms)
- Foolbox
- Robust Acc: 0.6 (13 ms)
- ART
- Robust Acc: 0.6 (13 ms)



### BIM

In [15]:
for model_name in model_list:
    print('Model: {}'.format(model_name))
    model = load_model(model_name, norm='Linf').to(device)

    print("- Torchattacks")
    atk = torchattacks.BIM(model, eps=8/255, alpha=2/255, steps=10)
    start = datetime.datetime.now()
    adv_images = atk(images, labels)
    end = datetime.datetime.now()
    acc = clean_accuracy(model, adv_images, labels,device)
    print('- Robust Acc: {} ({} ms)'.format(acc, int((end-start).total_seconds()*1000)))

    print("- Foolbox")
    fmodel = fb.PyTorchModel(model, bounds=(0, 1))
    atk = fb.attacks.LinfBasicIterativeAttack(abs_stepsize=2/255, steps=10, random_start=False)
    start = datetime.datetime.now()
    _, adv_images, _ = atk(fmodel, images.to('cuda:0'), labels.to('cuda:0'), epsilons=8/255)
    end = datetime.datetime.now()
    acc = clean_accuracy(model, adv_images, labels,device)
    print('- Robust Acc: {} ({} ms)'.format(acc, int((end-start).total_seconds()*1000)))

    print("- ART")
    classifier = PyTorchClassifier(model=model, clip_values=(0, 1),
                                   loss=nn.CrossEntropyLoss(),
                                   optimizer=optim.Adam(model.parameters(), lr=0.01),
                                   input_shape=(3, 32, 32), nb_classes=10)
    atk = evasion.BasicIterativeMethod(batch_size=50,
                                       estimator=classifier, eps=8/255,
                                       eps_step=2/255, max_iter=10)
    start = datetime.datetime.now()
    #adv_images = torch.tensor(atk.generate(images, labels)).to(device)
    adv_images_np = atk.generate(images.cpu().numpy(), labels.cpu().numpy())
    adv_images = torch.from_numpy(adv_images_np).to(device)
    end = datetime.datetime.now()
    acc = clean_accuracy(model, adv_images, labels,device)
    print('- Robust Acc: {} ({} ms)'.format(acc, int((end-start).total_seconds()*1000)))

    print()

Model: Standard
- Torchattacks
- Robust Acc: 0.0 (415 ms)
- Foolbox
- Robust Acc: 0.0 (537 ms)
- ART


PGD - Batches:   0%|          | 0/1 [00:00<?, ?it/s]

- Robust Acc: 0.0 (707 ms)

Model: Wong2020Fast
- Torchattacks
- Robust Acc: 0.4 (60 ms)
- Foolbox
- Robust Acc: 0.4 (99 ms)
- ART


PGD - Batches:   0%|          | 0/1 [00:00<?, ?it/s]

- Robust Acc: 0.4 (177 ms)



### PGD

In [17]:
for model_name in model_list:
    print('Model: {}'.format(model_name))
    model = load_model(model_name, norm='Linf').to(device)

    print("- Torchattacks")
    atk = torchattacks.PGD(model, eps=8/255, alpha=2/255, steps=10, random_start=False)
    start = datetime.datetime.now()
    adv_images = atk(images, labels)
    end = datetime.datetime.now()
    acc = clean_accuracy(model, adv_images, labels,device)
    print('- Robust Acc: {} ({} ms)'.format(acc, int((end-start).total_seconds()*1000)))

    print("- Foolbox")
    fmodel = fb.PyTorchModel(model, bounds=(0, 1))
    atk = fb.attacks.LinfPGD(abs_stepsize=2/255, steps=10, random_start=False)
    start = datetime.datetime.now()
    _, adv_images, _ = atk(fmodel, images.to('cuda:0'), labels.to('cuda:0'), epsilons=8/255)
    end = datetime.datetime.now()
    acc = clean_accuracy(model, adv_images, labels,device)
    print('- Robust Acc: {} ({} ms)'.format(acc, int((end-start).total_seconds()*1000)))

    print("- ART")
    classifier = PyTorchClassifier(model=model, clip_values=(0, 1),
                                   loss=nn.CrossEntropyLoss(),
                                   optimizer=optim.Adam(model.parameters(), lr=0.01),
                                   input_shape=(3, 32, 32), nb_classes=10)
    atk = evasion.ProjectedGradientDescent(batch_size=50, num_random_init=0,
                                           estimator=classifier, eps=8/255,
                                           eps_step=2/255, max_iter=10)
    start = datetime.datetime.now()
    adv_images_np = atk.generate(images.cpu().numpy(), labels.cpu().numpy())
    adv_images = torch.from_numpy(adv_images_np).to(device)
    end = datetime.datetime.now()
    acc = clean_accuracy(model, adv_images, labels,device)
    print('- Robust Acc: {} ({} ms)'.format(acc, int((end-start).total_seconds()*1000)))

    print()

Model: Standard
- Torchattacks
- Robust Acc: 0.0 (380 ms)
- Foolbox
- Robust Acc: 0.0 (546 ms)
- ART


PGD - Batches:   0%|          | 0/1 [00:00<?, ?it/s]

- Robust Acc: 0.0 (727 ms)

Model: Wong2020Fast
- Torchattacks
- Robust Acc: 0.4 (64 ms)
- Foolbox
- Robust Acc: 0.4 (97 ms)
- ART


PGD - Batches:   0%|          | 0/1 [00:00<?, ?it/s]

- Robust Acc: 0.4 (179 ms)



## 3.2. L2

## DeepFool

In [21]:
for model_name in model_list:
    print('Model: {}'.format(model_name))
    model = load_model(model_name, norm='Linf').to(device)

    print("- Torchattacks")
    atk = torchattacks.DeepFool(model, steps=50, overshoot=0.02)
    start = datetime.datetime.now()
    adv_images = atk(images, labels)
    end = datetime.datetime.now()
    acc = clean_accuracy(model, adv_images, labels,device)
    l2 = l2_distance(model, images, adv_images, labels, device=device)
    print('- Robust Acc: {} / L2: {:1.2} ({} ms)'.format(acc, l2,
                                                         int((end-start).total_seconds()*1000)))

    print("- Foolbox")
    fmodel = fb.PyTorchModel(model, bounds=(0, 1))
    atk = fb.attacks.L2DeepFoolAttack(steps=50, candidates=10, overshoot=0.02)
    start = datetime.datetime.now()
    _, adv_images, _ = atk(fmodel, images.to('cuda'), labels.to('cuda'), epsilons=1)
    end = datetime.datetime.now()
    acc = clean_accuracy(model, adv_images, labels,device)
    l2 = l2_distance(model, images, adv_images, labels, device=device)
    print('- Robust Acc: {} / L2: {:1.2} ({} ms)'.format(acc, l2,
                                                         int((end-start).total_seconds()*1000)))

    print("- ART")
    classifier = PyTorchClassifier(model=model, clip_values=(0, 1),
                                   loss=nn.CrossEntropyLoss(),
                                   optimizer=optim.Adam(model.parameters(), lr=0.01),
                                   input_shape=(3, 32, 32), nb_classes=10)
    atk = evasion.DeepFool(classifier=classifier, max_iter=50,
                           batch_size=50)

    start = datetime.datetime.now()
    adv_images_np = atk.generate(images.cpu().numpy(), labels.cpu().numpy())
    adv_images = torch.from_numpy(adv_images_np).to(device)
    end = datetime.datetime.now()
    acc = clean_accuracy(model, adv_images, labels,device)
    l2 = l2_distance(model, images, adv_images, labels, device=device)
    print('- Robust Acc: {} / L2: {:1.2} ({} ms)'.format(acc, l2,
                                                         int((end-start).total_seconds()*1000)))

    print()

Model: Standard
- Torchattacks
- Robust Acc: 0.0 / L2: 0.16 (4762 ms)
- Foolbox
- Robust Acc: 0.0 / L2: 0.16 (3229 ms)
- ART


DeepFool:   0%|          | 0/1 [00:00<?, ?it/s]

- Robust Acc: 0.2 / L2: 0.15 (25210 ms)

Model: Wong2020Fast
- Torchattacks
- Robust Acc: 0.0 / L2: 0.79 (885 ms)
- Foolbox
- Robust Acc: 0.4 / L2: 0.46 (415 ms)
- ART


DeepFool:   0%|          | 0/1 [00:00<?, ?it/s]

- Robust Acc: 0.1 / L2: 0.82 (5657 ms)



### CW

In [25]:
for model_name in model_list:
    print('Model: {}'.format(model_name))
    model = load_model(model_name, norm='Linf').to(device)

    print("- Torchattacks")
    atk = torchattacks.CW(model, c=1, kappa=0, steps=100, lr=0.01)
    start = datetime.datetime.now()
    adv_images = atk(images, labels)
    end = datetime.datetime.now()
    acc = clean_accuracy(model, adv_images, labels,device)
    l2 = l2_distance(model, images, adv_images, labels, device=device)
    print('- Robust Acc: {} / L2: {:1.2} ({} ms)'.format(acc, l2,
                                                         int((end-start).total_seconds()*1000)))

    print("- Foolbox")
    fmodel = fb.PyTorchModel(model, bounds=(0, 1))
    atk = fb.attacks.L2CarliniWagnerAttack(binary_search_steps=1, initial_const=1,
                                           confidence=0, steps=100, stepsize=0.01)
    start = datetime.datetime.now()
    _, adv_images, _ = atk(fmodel, images.to('cuda'), labels.to('cuda'), epsilons=1)
    end = datetime.datetime.now()
    acc = clean_accuracy(model, adv_images, labels,device)
    l2 = l2_distance(model, images, adv_images, labels, device=device)
    print('- Robust Acc: {} / L2: {:1.2} ({} ms)'.format(acc, l2,
                                                         int((end-start).total_seconds()*1000)))

    print("- ART")
    classifier = PyTorchClassifier(model=model, clip_values=(0, 1),
                                   loss=nn.CrossEntropyLoss(),
                                   optimizer=optim.Adam(model.parameters(), lr=0.01),
                                   input_shape=(3, 32, 32), nb_classes=10)
    atk = evasion.CarliniL2Method(batch_size=50, classifier=classifier,
                                  binary_search_steps=1, initial_const=1,
                                  confidence=0, max_iter=100,
                                  learning_rate=0.01)
    start = datetime.datetime.now()
    adv_images_np = atk.generate(images.cpu().numpy(), labels.cpu().numpy())
    adv_images = torch.from_numpy(adv_images_np).to(device)
    end = datetime.datetime.now()
    acc = clean_accuracy(model, adv_images, labels,device)
    l2 = l2_distance(model, images, adv_images, labels, device=device)
    print('- Robust Acc: {} / L2: {:1.2} ({} ms)'.format(acc, l2,
                                                         int((end-start).total_seconds()*1000)))

    print()

Model: Standard
- Torchattacks
- Robust Acc: 0.0 / L2: 0.34 (1456 ms)
- Foolbox
- Robust Acc: 0.0 / L2: 0.31 (1392 ms)
- ART


C&W L_2:   0%|          | 0/1 [00:00<?, ?it/s]

- Robust Acc: 0.0 / L2: 0.41 (64544 ms)

Model: Wong2020Fast
- Torchattacks
- Robust Acc: 0.0 / L2: 0.57 (1414 ms)
- Foolbox
- Robust Acc: 0.1 / L2: 0.5 (1516 ms)
- ART


C&W L_2:   0%|          | 0/1 [00:00<?, ?it/s]

- Robust Acc: 0.1 / L2: 0.67 (18082 ms)



### PGD L2

In [31]:
for model_name in model_list:
    print('Model: {}'.format(model_name))
    model = load_model(model_name, norm='Linf').cuda()

    print("- Torchattacks")
    atk = torchattacks.PGDL2(model, eps=128/255, alpha=15/255, steps=10, random_start=False)
    start = datetime.datetime.now()
    adv_images = atk(images, labels)
    end = datetime.datetime.now()
    acc = clean_accuracy(model, adv_images, labels,device)
    l2 = l2_distance(model, images, adv_images, labels, device=device)
    print('- Robust Acc: {} / L2: {:1.2} ({} ms)'.format(acc, l2,
                                                         int((end-start).total_seconds()*1000)))

    print("- Foolbox")
    fmodel = fb.PyTorchModel(model, bounds=(0, 1))
    atk = fb.attacks.L2PGD(abs_stepsize=15/255, steps=10, random_start=False)
    start = datetime.datetime.now()
    _, adv_images, _ = atk(fmodel, images.to('cuda:0'), labels.to('cuda:0'), epsilons=128/255)
    end = datetime.datetime.now()
    acc = clean_accuracy(model, adv_images, labels,device)
    l2 = l2_distance(model, images, adv_images, labels, device=device)
    print('- Robust Acc: {} / L2: {:1.2} ({} ms)'.format(acc, l2,
                                                         int((end-start).total_seconds()*1000)))

    print("- ART")
    classifier = PyTorchClassifier(model=model, clip_values=(0, 1),
                                   loss=nn.CrossEntropyLoss(),
                                   optimizer=optim.Adam(model.parameters(), lr=0.01),
                                   input_shape=(3, 32, 32), nb_classes=10)
    atk = evasion.ProjectedGradientDescent(batch_size=50, num_random_init=0,
                                           norm = 2, estimator=classifier, eps=128/255,
                                           eps_step=15/255, max_iter=10)
    start = datetime.datetime.now()
    adv_images_np = atk.generate(images.cpu().numpy(), labels.cpu().numpy())
    adv_images = torch.from_numpy(adv_images_np).to(device)
    end = datetime.datetime.now()
    acc = clean_accuracy(model, adv_images, labels,device)
    l2 = l2_distance(model, images, adv_images, labels, device=device)
    print('- Robust Acc: {} / L2: {:1.2} ({} ms)'.format(acc, l2,
                                                         int((end-start).total_seconds()*1000)))

    print()

Model: Standard
- Torchattacks
- Robust Acc: 0.0 / L2: 0.42 (413 ms)
- Foolbox
- Robust Acc: 0.0 / L2: 0.42 (569 ms)
- ART


PGD - Batches:   0%|          | 0/1 [00:00<?, ?it/s]

- Robust Acc: 0.0 / L2: 0.42 (738 ms)

Model: Wong2020Fast
- Torchattacks
- Robust Acc: 0.7 / L2: 0.5 (58 ms)
- Foolbox
- Robust Acc: 0.7 / L2: 0.5 (103 ms)
- ART


PGD - Batches:   0%|          | 0/1 [00:00<?, ?it/s]

- Robust Acc: 0.7 / L2: 0.5 (181 ms)



## 4. Comparison with AutoAttack

In [34]:
pip install git+https://github.com/fra31/auto-attack.git

  Cloning https://github.com/fra31/auto-attack.git to /tmp/pip-req-build-pytmii4_
  Running command git clone --filter=blob:none --quiet https://github.com/fra31/auto-attack.git /tmp/pip-req-build-pytmii4_
  Resolved https://github.com/fra31/auto-attack.git to commit a39220048b3c9f2cca9a4d3a54604793c68eca7e
  Preparing metadata (setup.py) ... done
  Created wheel for autoattack: filename=autoattack-0.1-py3-none-any.whl size=36228 sha256=8dccaaec2c58add19f6613b8b3cd738522e959dcf54244b44d3e3791c76a698e
  Stored in directory: /tmp/pip-ephem-wheel-cache-ivjn4gc1/wheels/d3/da/df/403e2ecb13ead4fe5da562006891405180c19c91db70e16d55
Successfully built autoattack


In [35]:
# https://github.com/fra31/auto-attack
from autoattack import AutoAttack

In [38]:
for model_name in model_list:
    print('Model: {}'.format(model_name))
    model = load_model(model_name, norm='Linf').to(device)

    print("- Torchattacks")
    atk = torchattacks.AutoAttack(model, eps=8/255)
    start = datetime.datetime.now()
    adv_images = atk(images, labels)
    end = datetime.datetime.now()
    acc = clean_accuracy(model, adv_images, labels,device)
    print('- Robust Acc: {} ({} ms)'.format(acc, int((end-start).total_seconds()*1000)))

    print("- Torchattacks")
    atk = AutoAttack(model, norm='Linf', eps=8/255, version='standard')
    start = datetime.datetime.now()
    adv_images = atk.run_standard_evaluation(images, labels, bs=len(images)).cuda()
    end = datetime.datetime.now()
    acc = clean_accuracy(model, adv_images, labels,device)
    print('- Robust Acc: {} ({} ms)'.format(acc, int((end-start).total_seconds()*1000)))

    print()

Model: Standard
- Torchattacks
- Robust Acc: 0.0 (679 ms)
- Torchattacks
setting parameters for standard version
using standard version including apgd-ce, apgd-t, fab-t, square.
initial accuracy: 100.00%
apgd-ce - 1/1 - 10 out of 10 successfully perturbed
robust accuracy after APGD-CE: 0.00% (total time 5.3 s)
max Linf perturbation: 0.03137, nan in tensor: 0, max: 1.00000, min: 0.00000
robust accuracy: 0.00%
- Robust Acc: 0.0 (5558 ms)

Model: Wong2020Fast
- Torchattacks
- Robust Acc: 0.3 (26257 ms)
- Torchattacks
setting parameters for standard version
using standard version including apgd-ce, apgd-t, fab-t, square.
initial accuracy: 100.00%
apgd-ce - 1/1 - 6 out of 10 successfully perturbed
robust accuracy after APGD-CE: 40.00% (total time 1.0 s)
apgd-t - 1/1 - 1 out of 4 successfully perturbed
robust accuracy after APGD-T: 30.00% (total time 9.5 s)
fab-t - 1/1 - 0 out of 3 successfully perturbed
robust accuracy after FAB-T: 30.00% (total time 26.7 s)
square - 1/1 - 0 out of 3 succes